# v2 Smoke — Blender + Flux Background Inpaint

Interactive smoke на RunPod A100/H100. Класс: **light_vehicle** (GAZ Tigr).

**Flow:** 1 frame → eyeball **Gate 1** → 5 frames → **Gate 2** → 20 frames batch → zip download.

**Контракт:** bbox/мітки незмінні. relight+degradation свідомо узгоджують яскравість/тон техніки під фон (пікселі техніки змінюються, геометрія — ні).

## ⚙️ SETTINGS — edit here, then re-run

**Усі ручки в одному місці.** Не треба чіпати YAML.

- Змінила **Flux / relight / degradation / промпт / `true_cfg_scale`** → re-run цей селл, потім **Cell 6 (inpaint) + Cell 7 (composite)**. Без reload pipeline, без re-render — швидко.
- Змінила **`lora_*`** (увімк/репо/scale) → LoRA вантажиться у Cell 5: re-run цей селл, потім **Cell 5 → 6 → 7**.
- Змінила **scene / road / landscapes** → re-render: re-run цей селл, потім **Cell 3** і далі.

⚠️ **`true_cfg_scale`**: FLUX-dev distilled — negative_prompt діє тільки при значенні >1 (2-4), і це ≈×2 час. При 1.0 негатив ігнорується (нормально, якщо тягне LoRA).

⚠️ bbox/мітки relight і degradation **не рухають** — тільки колір/яскравість пікселів.


In [ ]:
# ⚙️ SETTINGS — РЕДАГУЙ ТУТ
# Числа = стартові значення. Підказки по свіпу — у коментарях.

SETTINGS = {
    # --- Flux background ---
    'guidance': 3.5,          # ↓ якщо фон «пластик»/оверсат (3.5→3.0→2.5)
    'strength': 0.88,         # ↓ якщо втрачає масштаб/Blender-грунт (0.88→0.80)
    'steps': 30,              # ↑ 40-50 якщо фон надто blurred
    # negative_prompt діє ЛИШЕ при true_cfg_scale>1 (FLUX-dev distilled). ≈×2 час.
    'true_cfg_scale': 1.0,    # 1.0=негатив OFF; 2-4=негатив активний (дорожче)
    # realism-LoRA — найсильніший анти-«пластик» (сильніший за промпт).
    'lora_enabled': True,
    'lora_repo': 'XLabs-AI/flux-RealismLora',
    'lora_weight_name': 'lora.safetensors',
    'lora_scale': 0.8,        # 0.6-1.0; ↑ сильніше «фото», ↓ ближче до Flux-base

    # --- relight: підтягує яскравість/тон ТЕХНІКИ під фон (bbox не чіпає) ---
    'relight_enabled': True,
    'relight_strength': 0.55, # 0=без змін, 1=повністю під фон (0.4-0.7)
    'relight_match_color': True,  # per-channel WB+exposure; False = тільки яскравість

    # --- camera-realism degradation: 'погана' зйомка з дрона (на весь кадр) ---
    'degradation_enabled': True,
    'blur_sigma': 0.6,        # легкий defocus (0 = вимкнути)
    'motion_blur_px': 3,      # дрон рухається (0 = вимкнути)
    'chroma_px': 1.2,         # chromatic aberration дешевої оптики
    'atmosphere_strength': 0.10,  # aerial haze (0-0.6)
    'vignette_strength': 0.22,    # потемніння кутів
    'noise_sigma': 3.0,       # ↑ якщо замало «брудно/камерно» (0-8)
    'jpeg_quality': 72,       # ↓ = сильніша компресія-tell (50-85)

    # --- scene (зміна → потрібен re-render через Cell 3) ---
    'road_under_vehicle': True,   # колія під технікою (фікс «посеред поля»)
    'landscapes': ['dirt_road', 'field', 'dirt_road', 'forest_belt'],
}

# ─────────────────────────────────────────────────────────────────────────
# PROMPT — редагуй текст напряму. {плейсхолдери} підставляються з metadata кадру.
# Base template + cue нижче склеюються у фінальний prompt (drone EO, не кіно).
PROMPT_TEMPLATE = (
    'aerial drone reconnaissance photo of a {landscape} in {season}, {weather}, '
    '{sun_cardinal} sunlight at {sun_elevation_deg:.0f} degrees elevation, '
    'shot from {altitude_m:.0f} meters altitude at {view_angle_deg:.0f} degrees '
    'oblique top-down angle'
)

# Куди саме ставимо техніку — ключове проти «посеред поля».
LANDSCAPE_CUES = {
    'dirt_road': ('the vehicle is driving along a narrow dirt road, two parallel '
                  'tire ruts in packed earth running directly beneath and ahead of '
                  'the vehicle, dusty unpaved track'),
    'field': ('the vehicle sits on a faint farm track crossing an open field, '
              'flattened grass and tire marks under the vehicle'),
    'forest_belt': ('the vehicle is on a dirt track beside a tree line / forest '
                    'shelterbelt, track running under the vehicle'),
}

# Масштаб (aerial top-down дрібність рослинності).
SCALE_CUE = ('seen straight from above at high altitude, everything at true aerial '
             'scale, vegetation appears as tiny fine texture, no large foreground objects')

# Камера (amateur drone / EO sensor — НЕ cinematic).
CAMERA_CUE = ('low-resolution aerial surveillance footage, grainy telephoto drone '
              'shot, overcast flat lighting, slightly soft focus, visible sensor noise '
              'and jpeg compression artifacts, muted desaturated colors, candid '
              'unedited photo, amateur photo')

# Чого НЕ хочемо (анти-CGI/пластик/кіно).
NEGATIVE_PROMPT = (
    'person, soldier, vehicle, tank, truck, car, motorcycle, building, road sign, '
    'text, watermark, ui, 3d render, cgi, render, video game, unreal engine, octane, '
    'blender, plastic, glossy, waxy, airbrushed, cinematic, dramatic lighting, '
    'golden hour, lens flare, bokeh, depth of field, hdr, oversaturated, vivid, '
    'overprocessed, masterpiece, artstation, oversharpened, illustration, painting, '
    'blurry, low quality'
)

# ─────────────────────────────────────────────────────────────────────────
def _diffusion_overrides():
    return {
        'enabled': True,
        'guidance': SETTINGS['guidance'],
        'strength': SETTINGS['strength'],
        'steps': SETTINGS['steps'],
        'true_cfg_scale': SETTINGS['true_cfg_scale'],
        'lora': {'enabled': SETTINGS['lora_enabled'],
                 'repo': SETTINGS['lora_repo'],
                 'weight_name': SETTINGS['lora_weight_name'],
                 'adapter_name': 'realism',
                 'scale': SETTINGS['lora_scale']},
        'prompt_template': PROMPT_TEMPLATE,
        'negative_prompt': NEGATIVE_PROMPT,
        'landscape_cues': LANDSCAPE_CUES,
        'scale_cue': SCALE_CUE,
        'camera_cue': CAMERA_CUE,
        'relight': {'enabled': SETTINGS['relight_enabled'],
                    'strength': SETTINGS['relight_strength'],
                    'match_color': SETTINGS['relight_match_color']},
        'degradation': {'enabled': SETTINGS['degradation_enabled'],
                        'blur_sigma': SETTINGS['blur_sigma'],
                        'motion_blur_px': SETTINGS['motion_blur_px'],
                        'chroma_px': SETTINGS['chroma_px'],
                        'atmosphere_strength': SETTINGS['atmosphere_strength'],
                        'haze_airlight': 200,
                        'vignette_strength': SETTINGS['vignette_strength'],
                        'noise_sigma': SETTINGS['noise_sigma'],
                        'jpeg_quality': SETTINGS['jpeg_quality'],
                        'seed_offset': 7000},
    }

def apply_settings(cfg_dict):
    '''Merge SETTINGS у cfg_dict (для YAML, що пише Cell 3). Чіпає diffusion+scene.'''
    cfg_dict.setdefault('diffusion', {}).update(_diffusion_overrides())
    sc = cfg_dict.setdefault('scene', {})
    sc['road_under_vehicle'] = SETTINGS['road_under_vehicle']
    sc['landscapes'] = SETTINGS['landscapes']
    return cfg_dict

# Якщо diff_cfg вже завантажений (Cell 5) — оновлюємо in-place для швидкої
# ітерації: edit SETTINGS → re-run цей селл → re-run Cell 6 + 7 (без reload pipeline).
try:
    diff_cfg.update(_diffusion_overrides())
    print('✓ diff_cfg оновлено in-place — re-run Cell 6 (inpaint) + Cell 7 (composite)')
except NameError:
    print('✓ settings задані — далі по черзі Cell 1 → 2 → 3 …')


In [ ]:
# Cell 1 — env check
import os, sys, torch
print(f'Python: {sys.version.split()[0]}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'  CUDA: {torch.version.cuda}')
assert torch.cuda.is_available(), 'no CUDA — wrong pod template?'
assert torch.cuda.get_device_properties(0).total_memory >= 40e9, 'need >=40GB VRAM'

assert 'HF_TOKEN' in os.environ, "HF_TOKEN env var required (RunPod 'Pod env vars')"
print(f"HF_TOKEN: {os.environ['HF_TOKEN'][:8]}...")
print(f"HF_HOME: {os.environ.get('HF_HOME', '<default>')}")

from huggingface_hub import whoami
hf_user = whoami(token=os.environ['HF_TOKEN'])['name']
print(f'HF user: {hf_user}')
print('\n✓ env ready')

In [ ]:
# Cell 2 — repo + imports
import subprocess, sys
from pathlib import Path

REPO_DIR = Path('/workspace/yolo-bluebierd')
assert REPO_DIR.exists(), f'clone repo first: git clone <url> {REPO_DIR}'

subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=False)
subprocess.run(['pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements_diffusion.txt')], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from datasetforge.pipelines.inpaint import bg_filler, composite, prompts
print('✓ modules imported')

In [ ]:
# Cell 3 — Stage 1: render 1 frame at diffusion resolution
import os, subprocess, yaml
from pathlib import Path

REPO_DIR = Path('/workspace/yolo-bluebierd')
ASSETS = REPO_DIR / 'datasetforge/assets'
CFG_SRC = REPO_DIR / 'datasetforge/configs/v1_light_vehicle.yaml'
OUT = Path('/workspace/output/v2_smoke_1frame')
OUT.mkdir(parents=True, exist_ok=True)

cfg_dict = yaml.safe_load(CFG_SRC.read_text())
inf_size = cfg_dict.get('image_size_diffusion', [1024, 1024])
cfg_dict['image_size'] = inf_size
cfg_dict = apply_settings(cfg_dict)  # ← ручки з SETTINGS-селла вгорі

CFG_RUN = OUT / 'v1_light_vehicle_diff.yaml'
CFG_RUN.write_text(yaml.safe_dump(cfg_dict))
print(f'render @ {inf_size}, cfg → {CFG_RUN}')

env = os.environ.copy(); env['MPLBACKEND'] = 'agg'
cmd = ['blenderproc', 'run', str(REPO_DIR / 'datasetforge/engine/render_runner.py'),
       '--config', str(CFG_RUN), '--n', '1', '--out', str(OUT),
       '--assets-root', str(ASSETS), '--seed', '42']
subprocess.run(cmd, env=env, check=True)

print(f'\n✓ Stage 1 → {OUT}')
for sub in ['images','labels','depth','normals','vehicle_masks','metadata']:
    n = len(list((OUT / sub).glob('*')))
    print(f'  {sub}: {n} file(s)')

In [ ]:
# Cell 4 — Stage 1 preview (RGB+bbox / depth / normals / mask)
import json, cv2, numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path

OUT = Path('/workspace/output/v2_smoke_1frame')
stem = sorted(p.stem for p in (OUT / 'images').glob('*.jpg'))[0]
print(f'preview: {stem}')

rgb = np.array(Image.open(OUT/'images'/f'{stem}.jpg'))
depth = cv2.imread(str(OUT/'depth'/f'{stem}.png'), cv2.IMREAD_UNCHANGED)
normals_raw = cv2.imread(str(OUT/'normals'/f'{stem}.png'), cv2.IMREAD_UNCHANGED)
mask = cv2.imread(str(OUT/'vehicle_masks'/f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
meta = json.loads((OUT/'metadata'/f'{stem}.json').read_text())

H, W = rgb.shape[:2]
rgb_pil = Image.fromarray(rgb).copy()
d = ImageDraw.Draw(rgb_pil)
lbl = OUT / 'labels' / f'{stem}.txt'
if lbl.exists():
    for line in lbl.read_text().strip().splitlines():
        p = line.split(); xc, yc, w, h = map(float, p[1:5])
        d.rectangle([(xc-w/2)*W, (yc-h/2)*H, (xc+w/2)*W, (yc+h/2)*H], outline='lime', width=3)

normals_viz = ((normals_raw.astype(np.float32)/65535.0)*255).astype(np.uint8) if normals_raw.dtype==np.uint16 else normals_raw

fig, ax = plt.subplots(2, 2, figsize=(12, 12))
ax[0,0].imshow(rgb_pil); ax[0,0].set_title('RGB + bbox'); ax[0,0].axis('off')
ax[0,1].imshow(depth, cmap='viridis'); ax[0,1].set_title(f'Depth (mm) max={int(depth.max())}'); ax[0,1].axis('off')
ax[1,0].imshow(normals_viz); ax[1,0].set_title('Normals (encoded)'); ax[1,0].axis('off')
ax[1,1].imshow(mask, cmap='gray'); ax[1,1].set_title(f'Mask veh_px={int((mask>=128).sum())}'); ax[1,1].axis('off')
plt.tight_layout(); plt.show()

print(f"\nsun: {meta['sun_cardinal']} @ {meta['sun_elevation_deg']:.0f}° elev")
print(f"season={meta['season']} weather={meta['weather']} landscape={meta['landscape']}")
print(f"distance={meta['distance_m']}m angle={meta['view_angle_deg']}°")

In [ ]:
# Cell 5 — Load Flux pipeline (one-time, ~60-90s, ~30 GB VRAM)
import time, yaml, torch
from pathlib import Path
from datasetforge.pipelines.inpaint import bg_filler

CFG_RUN = Path('/workspace/output/v2_smoke_1frame/v1_light_vehicle_diff.yaml')
cfg_full = yaml.safe_load(CFG_RUN.read_text())
diff_cfg = cfg_full['diffusion']
print(f"pipeline: {diff_cfg['pipeline']}")
print(f"base:     {diff_cfg['base_model']}")
print(f"steps={diff_cfg['steps']} guidance={diff_cfg['guidance']} strength={diff_cfg.get('strength',1.0)}")
print(f"relight: {diff_cfg.get('relight',{})}")
print(f"degradation: {'on' if diff_cfg.get('degradation',{}).get('enabled') else 'off'}")

t0 = time.time()
pipe = bg_filler.load_pipeline(diff_cfg, device='cuda')
torch.cuda.synchronize()
print(f'\n✓ loaded in {time.time()-t0:.1f}s')
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f}/{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# Cell 6 — Stage 3: inpaint 1 frame
import time
from pathlib import Path
from datasetforge.pipelines.inpaint import bg_filler

OUT = Path('/workspace/output/v2_smoke_1frame')
stem = sorted(p.stem for p in (OUT/'images').glob('*.jpg'))[0]
(OUT/'ai_bg').mkdir(exist_ok=True)

t0 = time.time()
sidecar = bg_filler.inpaint_one(
    pipe,
    rgb_path=OUT/'images'/f'{stem}.jpg',
    depth_path=OUT/'depth'/f'{stem}.png',
    mask_path=OUT/'vehicle_masks'/f'{stem}.png',
    meta_path=OUT/'metadata'/f'{stem}.json',
    out_path=OUT/'ai_bg'/f'{stem}.png',
    diffusion_cfg=diff_cfg,
)
print(f'\n✓ inpaint {time.time()-t0:.1f}s | seed={sidecar["seed"]}')
print(f'\nprompt: {sidecar["prompt"]}')

In [ ]:
# Cell 7 — Stage 4: composite + 3-up preview
import numpy as np, cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path
from datasetforge.pipelines.inpaint import composite

OUT = Path('/workspace/output/v2_smoke_1frame')
stem = sorted(p.stem for p in (OUT/'images').glob('*.jpg'))[0]
(OUT/'composite').mkdir(exist_ok=True)

stats = composite.composite_one(
    rgb_path=OUT/'images'/f'{stem}.jpg',
    ai_bg_path=OUT/'ai_bg'/f'{stem}.png',
    mask_path=OUT/'vehicle_masks'/f'{stem}.png',
    normals_path=OUT/'normals'/f'{stem}.png',
    meta_path=OUT/'metadata'/f'{stem}.json',
    out_path=OUT/'composite'/f'{stem}.png',
    diffusion_cfg=diff_cfg,
    assert_pixel_identity=True,
)
print(f'✓ composite stats: {stats}')

rgb = np.array(Image.open(OUT/'images'/f'{stem}.jpg'))
ai_bg = np.array(Image.open(OUT/'ai_bg'/f'{stem}.png'))
comp = np.array(Image.open(OUT/'composite'/f'{stem}.png'))
mask = cv2.imread(str(OUT/'vehicle_masks'/f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
mb = mask >= 128
diff = int(np.abs(rgb.astype(int) - comp.astype(int))[mb].max())
_relit = diff_cfg.get('relight',{}).get('enabled')
_degr = diff_cfg.get('degradation',{}).get('enabled')
print(f'vehicle pixel max-diff: {diff} '
      f"({'expected 0 — both OFF' if not (_relit or _degr) else 'nonzero by design — relight/degradation harmonize the vehicle'})")

H, W = comp.shape[:2]
comp_pil = Image.fromarray(comp).copy()
d = ImageDraw.Draw(comp_pil)
lbl = OUT/'labels'/f'{stem}.txt'
if lbl.exists():
    for line in lbl.read_text().strip().splitlines():
        p = line.split(); xc, yc, w, h = map(float, p[1:5])
        d.rectangle([(xc-w/2)*W, (yc-h/2)*H, (xc+w/2)*W, (yc+h/2)*H], outline='lime', width=3)

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(rgb); ax[0].set_title('Stage 1 — raw 3D'); ax[0].axis('off')
ax[1].imshow(ai_bg); ax[1].set_title('Stage 3 — AI background'); ax[1].axis('off')
ax[2].imshow(comp_pil); ax[2].set_title('Stage 4 — composite (harmonized + camera-degraded)'); ax[2].axis('off')
plt.tight_layout(); plt.show()

## ⚖️ APPROVE GATE 1

Подивись на composite вище:
- ✓ Яскравість/тон техніки збігається з фоном (relight match_color)?
- ✓ Фон не «пластиковий», виглядає як зйомка з дрона (не кіно)?
- ✓ Масштаб рослинності відповідає aerial висоті?
- ✓ Техніка на дорозі/колії, а не посеред поля?
- ✓ Bbox tight навколо vehicle?

**Якщо ОК** → run Cell 9 (5 frames).

**Якщо НІ** — підкрутити у `diff_cfg` (потім re-run cells 6-7):
- Фон все ще «пластик» / оверсатурований → `guidance` 3.5 → 3.0 → 2.5
- Фон не в тому масштабі / втрачає Blender-грунт → `strength` 0.88 → 0.80
- Техніка все ще іншої яскравості → `relight.strength` 0.55 → 0.7
- Занадто «чисто/кіно» → підняти `degradation.noise_sigma` / знизити `jpeg_quality`
- AI bg занадто blurred → `steps` 40-50, `degradation.blur_sigma` → 0

In [ ]:
# Cell 9 — Scale до 5 frames (4 seasons × landscapes)
import time, subprocess, os
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from datasetforge.pipelines.inpaint import bg_filler, composite

REPO_DIR = Path('/workspace/yolo-bluebierd')
OUT5 = Path('/workspace/output/v2_smoke_5frame')
OUT5.mkdir(parents=True, exist_ok=True)
CFG = Path('/workspace/output/v2_smoke_1frame/v1_light_vehicle_diff.yaml')

env = os.environ.copy(); env['MPLBACKEND'] = 'agg'
subprocess.run(['blenderproc', 'run',
                str(REPO_DIR/'datasetforge/engine/render_runner.py'),
                '--config', str(CFG), '--n', '5', '--out', str(OUT5),
                '--assets-root', str(REPO_DIR/'datasetforge/assets'),
                '--seed', '100'], env=env, check=True)

(OUT5/'ai_bg').mkdir(exist_ok=True); (OUT5/'composite').mkdir(exist_ok=True)
stems = sorted(p.stem for p in (OUT5/'images').glob('*.jpg'))
t0 = time.time()
for stem in stems:
    bg_filler.inpaint_one(pipe,
        rgb_path=OUT5/'images'/f'{stem}.jpg', depth_path=OUT5/'depth'/f'{stem}.png',
        mask_path=OUT5/'vehicle_masks'/f'{stem}.png', meta_path=OUT5/'metadata'/f'{stem}.json',
        out_path=OUT5/'ai_bg'/f'{stem}.png', diffusion_cfg=diff_cfg)
    composite.composite_one(
        rgb_path=OUT5/'images'/f'{stem}.jpg', ai_bg_path=OUT5/'ai_bg'/f'{stem}.png',
        mask_path=OUT5/'vehicle_masks'/f'{stem}.png', normals_path=OUT5/'normals'/f'{stem}.png',
        meta_path=OUT5/'metadata'/f'{stem}.json', out_path=OUT5/'composite'/f'{stem}.png',
        diffusion_cfg=diff_cfg, assert_pixel_identity=True)
print(f'\n✓ 5-frame Stage 3+4 in {time.time()-t0:.1f}s')

fig, ax = plt.subplots(5, 3, figsize=(15, 25))
for r, stem in enumerate(stems):
    ax[r,0].imshow(np.array(Image.open(OUT5/'images'/f'{stem}.jpg'))); ax[r,0].set_title(f'{stem} raw')
    ax[r,1].imshow(np.array(Image.open(OUT5/'ai_bg'/f'{stem}.png'))); ax[r,1].set_title(f'{stem} ai_bg')
    ax[r,2].imshow(np.array(Image.open(OUT5/'composite'/f'{stem}.png'))); ax[r,2].set_title(f'{stem} composite')
    for c in range(3): ax[r,c].axis('off')
plt.tight_layout(); plt.show()

## ⚖️ APPROVE GATE 2

5 кадрів — різні seasons / landscapes / GAZ Tigr варіанти. Перевір:
- ✓ Кожен composite має vehicle pixels intact (assert_pixel_identity не впав)?
- ✓ Сезони виглядають **по-різному**?
- ✓ Жодних AI-tells: melted text, impossible architecture, vehicle 'floats', tiling artifacts?
- ✓ Тіні +/- кореляють з `sun_cardinal` у metadata?

**Якщо ОК** → Cell 11 (20-frame batch + regression test).

In [ ]:
# Cell 11 — 20-frame batch
import time, os, subprocess
from pathlib import Path
from datasetforge.pipelines.inpaint import bg_filler, composite

REPO_DIR = Path('/workspace/yolo-bluebierd')
OUT20 = Path('/workspace/output/v2_smoke_20frame')
OUT20.mkdir(parents=True, exist_ok=True)
CFG = Path('/workspace/output/v2_smoke_1frame/v1_light_vehicle_diff.yaml')

env = os.environ.copy(); env['MPLBACKEND'] = 'agg'
t_wall = time.time()
subprocess.run(['blenderproc', 'run',
                str(REPO_DIR/'datasetforge/engine/render_runner.py'),
                '--config', str(CFG), '--n', '20', '--out', str(OUT20),
                '--assets-root', str(REPO_DIR/'datasetforge/assets'),
                '--seed', '200'], env=env, check=True)

(OUT20/'ai_bg').mkdir(exist_ok=True); (OUT20/'composite').mkdir(exist_ok=True)
stems = sorted(p.stem for p in (OUT20/'images').glob('*.jpg'))
for stem in stems:
    bg_filler.inpaint_one(pipe,
        rgb_path=OUT20/'images'/f'{stem}.jpg', depth_path=OUT20/'depth'/f'{stem}.png',
        mask_path=OUT20/'vehicle_masks'/f'{stem}.png', meta_path=OUT20/'metadata'/f'{stem}.json',
        out_path=OUT20/'ai_bg'/f'{stem}.png', diffusion_cfg=diff_cfg)
    composite.composite_one(
        rgb_path=OUT20/'images'/f'{stem}.jpg', ai_bg_path=OUT20/'ai_bg'/f'{stem}.png',
        mask_path=OUT20/'vehicle_masks'/f'{stem}.png', normals_path=OUT20/'normals'/f'{stem}.png',
        meta_path=OUT20/'metadata'/f'{stem}.json', out_path=OUT20/'composite'/f'{stem}.png',
        diffusion_cfg=diff_cfg, assert_pixel_identity=True)

wall = time.time() - t_wall
GPU_USD_PER_HR = 1.7  # A100 80GB; H100 ≈ 2.5
print(f'\n✓ 20 frames in {wall:.0f}s ({wall/60:.1f} min)')
print(f'  est cost: ${wall/3600 * GPU_USD_PER_HR:.3f}')

In [ ]:
# Cell 12 — pack + download
import shutil
from pathlib import Path

OUT20 = Path('/workspace/output/v2_smoke_20frame')
zip_path = shutil.make_archive('/workspace/output/v2_smoke_20frame', 'zip',
                                root_dir=OUT20.parent, base_dir=OUT20.name)
size_mb = Path(zip_path).stat().st_size / 1e6
print(f'✓ {zip_path} ({size_mb:.1f} MB)')
print(f'\nDownload via RunPod web → File Browser → /workspace/output/v2_smoke_20frame.zip')

In [ ]:
# Cell 13 — vehicle-harmonization audit + co-occurrence sanity
import numpy as np, cv2, json, yaml
from PIL import Image
from pathlib import Path
from collections import Counter

OUT20 = Path('/workspace/output/v2_smoke_20frame')
CFG = Path('/workspace/output/v2_smoke_1frame/v1_light_vehicle_diff.yaml')
diff_cfg = yaml.safe_load(CFG.read_text())['diffusion']
_relit = bool(diff_cfg.get('relight',{}).get('enabled'))
_degr  = bool(diff_cfg.get('degradation',{}).get('enabled'))
stems = sorted(p.stem for p in (OUT20/'images').glob('*.jpg'))
print(f'validating {len(stems)} frames | relight={_relit} degradation={_degr}')

drifts = []
for stem in stems:
    rgb = np.array(Image.open(OUT20/'images'/f'{stem}.jpg'))
    comp = np.array(Image.open(OUT20/'composite'/f'{stem}.png'))
    mask = cv2.imread(str(OUT20/'vehicle_masks'/f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
    mb = mask >= 128
    if mb.any():
        drifts.append(int(np.abs(rgb.astype(int) - comp.astype(int))[mb].max()))

if not (_relit or _degr):
    bad = [d for d in drifts if d != 0]
    print('❌ %d frames broke byte-freeze' % len(bad) if bad
          else f'✓ all {len(stems)} frames: vehicle pixels byte-identical')
else:
    import statistics
    print(f'ℹ vehicle harmonized by design — drift min/med/max: '
          f'{min(drifts)}/{int(statistics.median(drifts))}/{max(drifts)}')
    print('  (expected nonzero: relight/degradation match the vehicle to the bg)')

pairs = Counter()
for stem in stems:
    m = json.loads((OUT20/'metadata'/f'{stem}.json').read_text())
    pairs[(m['season'], m['landscape'])] += 1
print('\n(season, landscape) distribution:')
for pair, n in pairs.most_common():
    pct = 100*n/len(stems)
    mark = '⚠' if pct > 35 else ' '
    print(f'  {mark} {pair}: {n} ({pct:.0f}%)')

## 🛑 Cleanup reminder

Pod продовжує білингитись до явної зупинки.

```bash
runpodctl pod stop <POD_ID>
```

Або через web UI → Pods → Stop.

**Після smoke — ротейтнути RunPod API key** (Settings → API Keys → Revoke + Create new).
**Після smoke — ротейтнути HF token** якщо хочеш бути параноїдальною.